In [1]:
from pydantic import BaseModel
import boto3
import json
import base64
import os
session = boto3.Session()

In [2]:
# Initialize clients
region = session.region_name
sagemaker_client = boto3.client("sagemaker", region_name=region)
runtime_client = boto3.client("sagemaker-runtime", region_name=region)
sts_client = boto3.client("sts", region_name=region)

In [3]:
model_name = "gemma-4-receipt-extraction-vllm"
endpoint_config_name = model_name
endpoint_name = model_name
account_id = sts_client.get_caller_identity()['Account']

In [4]:
container_image = f"{account_id}.dkr.ecr.us-west-2.amazonaws.com/vllm-gemma-4:0.19.1-sagemaker"
huggingface_model_id = "google/gemma-4-E4B-it"
instance_type = "ml.g6.2xlarge"
execution_role = f"arn:aws:iam::{account_id}:role/service-role/AmazonSageMaker-ExecutionRole-20260425T192955"

In [5]:
print(f"Creating SageMaker model: {model_name}")

create_model_response = sagemaker_client.create_model(
    ModelName=model_name,
    PrimaryContainer={
        'Image': container_image,
        'Environment': {
            'SM_VLLM_MODEL': huggingface_model_id
        }
    },
    ExecutionRoleArn=execution_role
)

print("Model created")

Creating SageMaker model: gemma-4-receipt-extraction-vllm


Model created


In [6]:
print(f"Creating endpoint configuration: {endpoint_config_name}")

create_endpoint_config_response = sagemaker_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            'VariantName': 'AllTraffic',
            'ModelName': model_name,
            'InstanceType': instance_type,
            'InitialInstanceCount': 1
        }
    ]
)

print("Endpoint configuration created")

Creating endpoint configuration: gemma-4-receipt-extraction-vllm


Endpoint configuration created


In [7]:
print(f"Creating endpoint: {endpoint_name}")

create_endpoint_response = sagemaker_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name
)

print(f"Monitor progress: https://console.aws.amazon.com/sagemaker/home?region={region}#/endpoints/{endpoint_name}\n")

Creating endpoint: gemma-4-receipt-extraction-vllm


Monitor progress: https://console.aws.amazon.com/sagemaker/home?region=us-west-2#/endpoints/gemma-4-receipt-extraction-vllm



In [8]:
class receiptExtraction(BaseModel):
    storeName: str
    purchaseDate: str
    total: float


def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode('utf-8')
    ext = os.path.splitext(image_path)[1].lower()
    mime_types = {
        '.jpg': 'image/jpeg',
        '.jpeg': 'image/jpeg',
        '.png': 'image/png'
    }
    mime_type = mime_types.get(ext, mime_types)
    return base64_image, mime_type


image_path = "photos/oakstreet.jpg"
base64_image, mime_type = encode_image(image_path)

PROMPT = """
You are a receipt extraction assistant.

Write from this receipt then must careful, do not fabricate and return JSON with this fields:
- storeName: in UPPERCASE format only.
- purchaseDate: in DD-MM-YYYY format only. If month not number (word), convert to number. Example : 10/12/2023 to 10/12/2023 NOT 12/10/2023 .
- total: total amount as float only, no currency symbol, no comma.

Do NOT add explanation or markdown.
"""

payload = {
    "messages": [
        {
            "role": "system",
            "content": PROMPT
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:{mime_type};base64,{base64_image}"
                    }
                }
            ]
        }
    ],
    "extra_body": {
        "chat_template_kwargs": {"enable_thinking": True}
    },
    "response_format": {
        "type": "json_schema",
        "json_schema": {
            "name": "receipt-extraction",
            "schema": receiptExtraction.model_json_schema()
        }
    }
}

response = runtime_client.invoke_endpoint(
    EndpointName="gemma-4-receipt-extraction-vllm",
    ContentType='application/json',
    Body=json.dumps(payload)
)

response_body = json.loads(response['Body'].read().decode())
message = response_body['choices'][0]['message']['content']
print(message)

{"storeName": "OAK STREET MARKET", "purchaseDate": "26-10-2023", "total": 42.58}


In [9]:
# Delete endpoint
sagemaker_client.delete_endpoint(EndpointName=endpoint_name)
print("Endpoint deleted")

# Delete endpoint configuration
print(f"\nDeleting endpoint configuration: {endpoint_config_name}")
sagemaker_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
print("Endpoint configuration deleted")

# Delete model
print(f"\nDeleting model: {model_name}")
sagemaker_client.delete_model(ModelName=model_name)
print("Model deleted")

Endpoint deleted

Deleting endpoint configuration: gemma-4-receipt-extraction-vllm
Endpoint configuration deleted

Deleting model: gemma-4-receipt-extraction-vllm


Model deleted
